# Hugging Face Pipeline Inference Tutorial

This notebook demonstrates how to use the Hugging Face `pipeline()` function for quick and easy inference with language models. The pipeline API is a high-level interface that simplifies working with pre-trained models for various NLP tasks.

## What is the Pipeline API?

The `pipeline()` function is a simple way to use models from the Hugging Face Hub for inference. It handles:
- Loading the model and tokenizer
- Pre-processing inputs
- Running the model
- Post-processing outputs

This makes it perfect for quickly implementing inference in applications.

## Setup Environment

First, let's install the required packages:

In [ ]:
!pip install -q transformers>=4.30.0 torch datasets nltk

## Import Libraries

In [ ]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import nltk
from nltk.corpus import wordnet

# Download WordNet for thesaurus functionality
nltk.download('wordnet')
nltk.download('omw-1.4')

## 1. Basic Pipeline Usage

Let's start with the simplest way to use the pipeline API:

In [ ]:
# Create a text generation pipeline with a small model
generator = pipeline('text-generation', model='distilgpt2')

# Generate text
prompt = "The synonym of happy is"
results = generator(prompt, max_length=30, num_return_sequences=3)

# Display results
for result in results:
    print(result['generated_text'])
    print("-" * 50)

## 2. Using Pipeline with a Fine-Tuned Model

If you have a fine-tuned model (e.g., from the LoRA or QLoRA tutorials), you can use it with the pipeline API:

In [ ]:
# This code assumes you have a fine-tuned model
# If you don't, you can comment this out and proceed with the next sections

# Check if the fine-tuned model exists
import os
if os.path.exists('./thesaurus-model-merged'):
    # Load the fine-tuned model
    fine_tuned_generator = pipeline(
        'text-generation',
        model='./thesaurus-model-merged',
        device=0 if torch.cuda.is_available() else -1
    )
    
    # Test with some words
    test_words = ["happy", "intelligent", "beautiful"]
    
    for word in test_words:
        prompt = f"### Instruction: List synonyms for the word '{word}'\n\n### Response:"
        results = fine_tuned_generator(prompt, max_length=100, temperature=0.7)
        
        # Extract the response
        response = results[0]['generated_text']
        response_parts = response.split("### Response:")
        if len(response_parts) > 1:
            synonyms = response_parts[1].strip()
        else:
            synonyms = response
        
        print(f"Synonyms for '{word}': {synonyms}")
        print("-" * 50)
else:
    print("Fine-tuned model not found. Continuing with pre-trained models.")

## 3. Using Different Pipeline Tasks

The pipeline API supports many different tasks. Let's explore some that are relevant for a thesaurus application:

In [ ]:
# Text classification pipeline
classifier = pipeline('sentiment-analysis')
results = classifier([
    "I am happy today!", 
    "I am feeling sad and depressed.",
    "The weather is nice."
])

print("Sentiment Analysis Results:")
for result in results:
    print(f"Label: {result['label']}, Score: {result['score']:.4f}")

print("\n" + "-" * 50 + "\n")

# Fill-mask pipeline
unmasker = pipeline('fill-mask', model='bert-base-uncased')
results = unmasker("The synonym of happy is [MASK].")

print("Fill-Mask Results:")
for result in results[:5]:  # Show top 5 results
    print(f"Token: {result['token_str']}, Score: {result['score']:.4f}")

print("\n" + "-" * 50 + "\n")

# Question answering pipeline
qa_pipeline = pipeline('question-answering')
context = """Synonyms are words that have the same or nearly the same meaning as another word. 
Antonyms are words that have the opposite meaning of another word. 
Homonyms are words that sound alike or are spelled alike but have different meanings."""

result = qa_pipeline({
    'question': 'What are synonyms?',
    'context': context
})

print("Question Answering Results:")
print(f"Answer: {result['answer']}")
print(f"Score: {result['score']:.4f}")

## 4. Creating a Custom Thesaurus Pipeline

Now, let's create a custom pipeline that combines WordNet and a language model to provide comprehensive thesaurus functionality:

In [ ]:
class ThesaurusLLM:
    def __init__(self, model_name="gpt2", device=None):
        """
        Initialize the ThesaurusLLM with a model.
        
        Args:
            model_name (str): Name or path of the model
            device (str, optional): Device to run the model on ('cpu', 'cuda', etc.)
        """
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        
        self.device = device
        self.generator = pipeline(
            "text-generation",
            model=model_name,
            device=0 if device == 'cuda' else -1
        )
    
    def get_synonyms_from_wordnet(self, word):
        """
        Get synonyms for a word from WordNet.
        
        Args:
            word (str): The word to find synonyms for
            
        Returns:
            list: List of synonyms
        """
        synonyms = []
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                if lemma.name() != word and lemma.name() not in synonyms:
                    synonyms.append(lemma.name().replace('_', ' '))
        return synonyms[:10]  # Return top 10 synonyms
    
    def get_antonyms_from_wordnet(self, word):
        """
        Get antonyms for a word from WordNet.
        
        Args:
            word (str): The word to find antonyms for
            
        Returns:
            list: List of antonyms
        """
        antonyms = []
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                if lemma.antonyms():
                    for antonym in lemma.antonyms():
                        if antonym.name() not in antonyms:
                            antonyms.append(antonym.name().replace('_', ' '))
        return antonyms
    
    def get_synonyms_from_llm(self, word, max_length=50, temperature=0.7):
        """
        Get synonyms for a word from the language model.
        
        Args:
            word (str): The word to find synonyms for
            max_length (int): Maximum length of generated text
            temperature (float): Temperature for text generation
            
        Returns:
            list: List of synonyms
        """
        prompt = f"List 5 synonyms for the word '{word}':\n1."
        
        results = self.generator(
            prompt,
            max_length=max_length,
            temperature=temperature,
            num_return_sequences=1,
            return_full_text=True
        )
        
        # Extract the generated synonyms
        generated_text = results[0]['generated_text']
        lines = generated_text.split('\n')
        
        synonyms = []
        for line in lines[1:]:  # Skip the prompt line
            if line.strip() and any(c.isdigit() for c in line[:2]):
                # Extract the synonym after the number and period
                parts = line.split('.', 1)
                if len(parts) > 1:
                    synonym = parts[1].strip()
                    if synonym and synonym != word:
                        synonyms.append(synonym)
        
        return synonyms
    
    def get_all_synonyms(self, word):
        """
        Get synonyms from both WordNet and the language model.
        
        Args:
            word (str): The word to find synonyms for
            
        Returns:
            dict: Dictionary with WordNet and LLM synonyms
        """
        wordnet_synonyms = self.get_synonyms_from_wordnet(word)
        llm_synonyms = self.get_synonyms_from_llm(word)
        
        # Combine and deduplicate
        all_synonyms = list(set(wordnet_synonyms + llm_synonyms))
        
        return {
            'word': word,
            'wordnet_synonyms': wordnet_synonyms,
            'llm_synonyms': llm_synonyms,
            'all_synonyms': all_synonyms,
            'antonyms': self.get_antonyms_from_wordnet(word)
        }

## 5. Testing the Custom Thesaurus Pipeline

In [ ]:
# Initialize the ThesaurusLLM
thesaurus_llm = ThesaurusLLM()

# Test with some words
test_words = ["happy", "intelligent", "beautiful", "strong"]

for word in test_words:
    print(f"\nThesaurus results for '{word}':")
    results = thesaurus_llm.get_all_synonyms(word)
    
    print(f"WordNet Synonyms: {', '.join(results['wordnet_synonyms'])}")
    print(f"LLM Synonyms: {', '.join(results['llm_synonyms'])}")
    print(f"Antonyms: {', '.join(results['antonyms'])}")
    print("-" * 50)

## 6. Using the Pipeline API with Different Models

Let's compare the results from different models:

In [ ]:
# List of models to compare
models = [
    "distilgpt2",  # Small GPT-2 model
    "gpt2",        # Base GPT-2 model
    "EleutherAI/pythia-410m"  # Pythia model
]

# Word to test
test_word = "happy"

for model_name in models:
    print(f"\nTesting model: {model_name}")
    
    # Create a generator with the model
    generator = pipeline(
        "text-generation",
        model=model_name,
        device=0 if torch.cuda.is_available() else -1
    )
    
    # Generate synonyms
    prompt = f"List 5 synonyms for the word '{test_word}':\n1."
    results = generator(
        prompt,
        max_length=50,
        temperature=0.7,
        num_return_sequences=1
    )
    
    print(f"Generated text:\n{results[0]['generated_text']}")
    print("-" * 50)

## 7. Advanced Pipeline Configuration

Let's explore some advanced configuration options for the pipeline API:

In [ ]:
# Load a model and tokenizer explicitly
model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Create a pipeline with the loaded model and tokenizer
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# Configure generation parameters
test_word = "intelligent"
prompt = f"List synonyms for the word '{test_word}':"

# Test different generation strategies
print("\nGreedy decoding (no randomness):")
results = generator(
    prompt,
    max_length=50,
    do_sample=False,  # Greedy decoding
    num_return_sequences=1
)
print(results[0]['generated_text'])

print("\nSampling with temperature:")
results = generator(
    prompt,
    max_length=50,
    do_sample=True,
    temperature=0.7,  # Lower = less random, higher = more random
    num_return_sequences=1
)
print(results[0]['generated_text'])

print("\nBeam search:")
results = generator(
    prompt,
    max_length=50,
    num_beams=5,  # Number of beams for beam search
    no_repeat_ngram_size=2,  # Avoid repeating 2-grams
    num_return_sequences=1
)
print(results[0]['generated_text'])

## 8. Integrating the Pipeline into a Web Application

Here's how you would integrate the pipeline into a Flask web application:

In [ ]:
# This is a code snippet that would be used in a Flask application
# It's not meant to be run in this notebook

'''
from flask import Flask, request, jsonify
from transformers import pipeline
import torch
import nltk
from nltk.corpus import wordnet

app = Flask(__name__)

# Initialize the pipeline globally
device = 'cuda' if torch.cuda.is_available() else 'cpu'
generator = pipeline(
    "text-generation",
    model="./thesaurus-model-merged",  # Path to your fine-tuned model
    device=0 if device == 'cuda' else -1
)

# Download WordNet
nltk.download('wordnet')
nltk.download('omw-1.4')

@app.route('/api/synonyms/<word>')
def get_synonyms(word):
    # Get synonyms from WordNet
    wordnet_synonyms = []
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            if lemma.name() != word and lemma.name() not in wordnet_synonyms:
                wordnet_synonyms.append(lemma.name().replace('_', ' '))
    
    # Get synonyms from the language model
    prompt = f"### Instruction: List synonyms for the word '{word}'\n\n### Response:"
    results = generator(
        prompt,
        max_length=100,
        temperature=0.7,
        num_return_sequences=1
    )
    
    # Extract the response
    response = results[0]['generated_text']
    response_parts = response.split("### Response:")
    if len(response_parts) > 1:
        llm_synonyms = response_parts[1].strip().split(', ')
    else:
        llm_synonyms = []
    
    # Combine and deduplicate
    all_synonyms = list(set(wordnet_synonyms + llm_synonyms))
    
    return jsonify({
        'word': word,
        'synonyms': all_synonyms[:10]  # Return top 10 synonyms
    })

if __name__ == '__main__':
    app.run(debug=True)
'''

## Conclusion

In this tutorial, you've learned how to:

1. Use the Hugging Face `pipeline()` function for quick inference
2. Work with different pipeline tasks (text generation, sentiment analysis, etc.)
3. Create a custom thesaurus pipeline that combines WordNet and language models
4. Compare results from different models
5. Configure advanced generation parameters
6. Integrate the pipeline into a web application

The pipeline API makes it easy to implement NLP functionality in your applications without having to worry about the low-level details of model inference.

## References

- [Hugging Face Pipelines Documentation](https://huggingface.co/docs/transformers/main_classes/pipelines)
- [Text Generation Strategies](https://huggingface.co/blog/how-to-generate)
- [NLTK WordNet Documentation](https://www.nltk.org/howto/wordnet.html)
- [Hugging Face Model Hub](https://huggingface.co/models)